In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pickle
import numpy as np

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [3]:
BATCH_SIZE = 1024
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

# Load mappings to get the total number of items for our Embedding layer
with open('mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)

# +1 because our item indices are 1-based, and 0 is reserved for padding
NUM_ITEMS = len(mappings['item2idx']) + 1 
print(f"Total items (including padding token): {NUM_ITEMS}")


Total items (including padding token): 59048


In [4]:
print("Loading processed datasets...")

train_data = torch.load('train_data.pt')
val_data = torch.load('val_data.pt')
test_data = torch.load('test_data.pt')

# Create PyTorch TensorDatasets
train_dataset = TensorDataset(train_data['X'], train_data['y'])
val_dataset = TensorDataset(val_data['X'], val_data['y'])
test_dataset = TensorDataset(test_data['X'], test_data['y'])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=4, pin_memory=True) # Added optimizations
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                         num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


Loading processed datasets...
Train batches: 23938
Val batches: 159


In [5]:
import torch
import torch.nn as nn
import math

class SASRec(nn.Module):
    def __init__(self, num_items, max_seq_len, embed_dim=64, num_heads=2, hidden_dim=128, num_blocks=2, dropout=0.2):
        super(SASRec, self).__init__()
        
        # 1. Item and Positional Embeddings
        self.item_emb = nn.Embedding(num_items, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.emb_dropout = nn.Dropout(dropout)
        
        # 2. Transformer Blocks
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=hidden_dim, 
            dropout=dropout, 
            activation='relu',
            batch_first=True # Ensures input shape is (batch, seq_len, features)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)
        
        # 3. Prediction Head
        self.out = nn.Linear(embed_dim, num_items)

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        batch_size, seq_len = x.size()
        
        # --- Embeddings ---
        # Create position indices: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
        
        item_embeddings = self.item_emb(x)
        pos_embeddings = self.pos_emb(positions)
        
        # Combine embeddings
        x_emb = item_embeddings + pos_embeddings
        x_emb = self.emb_dropout(x_emb)
        
        # --- Masks ---
        # 1. Padding mask: Ignore padded zeros (True means ignore)
        padding_mask = (x == 0)
        
        # 2. Causal mask: Prevent attending to future items
        # nn.Transformer.generate_square_subsequent_mask creates a mask with -inf for future tokens
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)
        
        # --- Transformer Forward ---
        transformer_out = self.transformer_encoder(
            x_emb, 
            mask=causal_mask, 
            src_key_padding_mask=padding_mask
        )
        
        # --- Prediction ---
        # We only care about the output at the very last time step to predict the NEXT item
        last_hidden_state = transformer_out[:, -1, :] # shape: (batch_size, embed_dim)
        
        # Calculate logits for all items
        logits = self.out(last_hidden_state)
        
        return logits

# Instantiate the SASRec model
# Assuming MAX_SEQ_LENGTH is 20 (from your data prep) and NUM_ITEMS is from mappings.pkl
MAX_SEQ_LENGTH = 20

model = SASRec(
    num_items=NUM_ITEMS, 
    max_seq_len=MAX_SEQ_LENGTH, 
    embed_dim=EMBEDDING_DIM, 
    hidden_dim=HIDDEN_DIM,
    num_heads=2,
    num_blocks=2
).to(device)

print(model)


SASRec(
  (item_emb): Embedding(59048, 64, padding_idx=0)
  (pos_emb): Embedding(20, 64)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (out): Linear(in_features=64, out_features=59048, bias=True)
)


In [7]:
import time
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

NUM_EPOCHS = 10
PATIENCE = 3
# Ensure you have NUM_ITEMS defined from your mappings (e.g., len(item_mapping) + 1)
# NUM_ITEMS = ... 

# 1. Changed to Binary Cross Entropy Loss
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 2. Initialize the AMP GradScaler
scaler = GradScaler()

# Early stopping variables
best_val_loss = float('inf')
patience_counter = 0
best_model_path = 'best_sasrec_model.pth'

print(f"Starting training for {NUM_EPOCHS} epochs with early stopping (Patience: {PATIENCE})...")

for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    
    model.train()
    total_train_loss = 0.0
    
    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False)
    for batch_X, batch_y in train_iterator:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        # 3. Generate negative samples (random movies the user didn't interact with)
        # Assuming item IDs start from 1 (0 is padding)
        neg_y = torch.randint(1, NUM_ITEMS, batch_y.size(), device=device)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # 4. Run forward pass with Mixed Precision
        with autocast():
            # Get logits from model
            logits = model(batch_X) 
            
            # Extract only the probabilities of the positive targets and negative targets
            # unsqueeze(-1) and squeeze(-1) ensures this works for both 1D and 2D target tensors
            pos_logits = logits.gather(-1, batch_y.unsqueeze(-1)).squeeze(-1)
            neg_logits = logits.gather(-1, neg_y.unsqueeze(-1)).squeeze(-1)
            
            # Create target labels (1 for true next movie, 0 for random negative movie)
            pos_targets = torch.ones_like(pos_logits)
            neg_targets = torch.zeros_like(neg_logits)
            
            # Combine them for the loss function
            final_logits = torch.cat([pos_logits, neg_logits], dim=0)
            final_targets = torch.cat([pos_targets, neg_targets], dim=0)
            
            # Calculate BCE Loss
            loss = criterion(final_logits, final_targets)
        
        # 5. Scale loss and backpropagate (AMP)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item() * batch_X.size(0)
        
        # Update progress bar with current batch loss
        train_iterator.set_postfix(loss=f"{loss.item():.4f}")
        
    avg_train_loss = total_train_loss / len(train_loader.dataset)
    
    # Validation 
    model.eval()
    total_val_loss = 0.0
    
    val_iterator = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False)
    with torch.no_grad():
        for batch_X, batch_y in val_iterator:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            # Generate negative samples for validation
            neg_y = torch.randint(1, NUM_ITEMS, batch_y.size(), device=device)
            
            with autocast():
                logits = model(batch_X)
                
                pos_logits = logits.gather(-1, batch_y.unsqueeze(-1)).squeeze(-1)
                neg_logits = logits.gather(-1, neg_y.unsqueeze(-1)).squeeze(-1)
                
                pos_targets = torch.ones_like(pos_logits)
                neg_targets = torch.zeros_like(neg_logits)
                
                final_logits = torch.cat([pos_logits, neg_logits], dim=0)
                final_targets = torch.cat([pos_targets, neg_targets], dim=0)
                
                loss = criterion(final_logits, final_targets)
            
            total_val_loss += loss.item() * batch_X.size(0)
            
    avg_val_loss = total_val_loss / len(val_loader.dataset)
    
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Time: {epoch_time:.2f}s | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Early Stopping 
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Save the best model
        torch.save(model.state_dict(), best_model_path)
        print("  --> Val loss improved, saving model.")
    else:
        patience_counter += 1
        print(f"  --> No improvement. Patience: {patience_counter}/{PATIENCE}")
        
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch+1} epochs!")
        break

# Load the best model weights back before moving on to testing
print("\nLoading the best model weights...")
model.load_state_dict(torch.load(best_model_path))
print("Done. Ready for evaluation.")


/tmp/ipykernel_857/1400349002.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Starting training for 10 epochs with early stopping (Patience: 3)...


/tmp/ipykernel_857/1400349002.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_857/1400349002.py:90: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
                                                        

Epoch 01/10 | Time: 591.52s | Train Loss: 0.1527 | Val Loss: 0.1225
  --> Val loss improved, saving model.


Epoch 02/10 | Time: 594.32s | Train Loss: 0.1229 | Val Loss: 0.1137
  --> Val loss improved, saving model.


...

Epoch 07/10 | Time: 589.93s | Train Loss: 0.1027 | Val Loss: 0.0997
  --> Val loss improved, saving model.


Epoch 08/10 | Time: 591.87s | Train Loss: 0.1012 | Val Loss: 0.0990
  --> Val loss improved, saving model.


Epoch 09/10 | Time: 595.53s | Train Loss: 0.0999 | Val Loss: 0.0973
  --> Val loss improved, saving model.


IOPub message rate exceeded.218/23938 [07:33<02:08, 44.49it/s, loss=0.1004]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

                                                                           

Epoch 10/10 | Time: 596.11s | Train Loss: 0.0988 | Val Loss: 0.0986
  --> No improvement. Patience: 1/3

Loading the best model weights...
Done. Ready for evaluation.


In [8]:
# Test Evaluation (Hit Rate @ 10)
def evaluate_hit_rate(model, data_loader, top_k=10):
    model.eval()
    hits = 0
    total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            # Get model predictions
            logits = model(batch_X)
            
            # Get the top K item indices
            # logits shape: (batch_size, num_items)
            _, top_k_indices = torch.topk(logits, top_k, dim=1)
            
            # Check if the true target (batch_y) is in the top K predictions
            # Expand batch_y to match dimensions for comparison
            batch_y_expanded = batch_y.unsqueeze(1).expand_as(top_k_indices)
            
            # Count hits in this batch
            hits += (top_k_indices == batch_y_expanded).sum().item()
            total += batch_y.size(0)
            
    return hits / total

# Run evaluation on the test set
test_hit_rate = evaluate_hit_rate(model, test_loader, top_k=10)
print(f"Test Hit Rate @ 10: {test_hit_rate:.4f} ({(test_hit_rate*100):.2f}%)")


Test Hit Rate @ 10: 0.0589 (5.89%)


In [10]:

import os

bundle_path = 'sasrec_model_bundle.pth'

# We package the architecture parameters and the trained state dict together
save_bundle = {
    'model_state_dict': model.state_dict(),
    'hyperparameters': {
        'NUM_ITEMS': NUM_ITEMS,
        'MAX_SEQ_LENGTH': MAX_SEQ_LENGTH,
        'EMBEDDING_DIM': EMBEDDING_DIM,
        'HIDDEN_DIM': HIDDEN_DIM
    }
}

torch.save(save_bundle, bundle_path)
print(f"Model bundle saved successfully to {bundle_path}")



Model bundle saved successfully to sasrec_model_bundle.pth
